<a href="https://colab.research.google.com/github/Jbrr2021/gcp-finops-python-automation/blob/main/classificacao_alertas_custo_pandas_dataframe_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐍 Rotulagem Condicional de Alertas Financeiros com Pandas

## 🗂️ Conceito Técnico: Uso do Pandas (.apply e Lambda) para Criação de Colunas Dinâmicas

### 💼 Contexto de Negócio (Pedido do Gestor):
> *"Para apoiar a tomada de decisões na fase de Otimização de FinOps, preciso que o nosso script em Python aplique uma regra de governança automatizada no DataFrame antes da exportação. Adicione uma coluna chamada 'status_orcamento': se o recurso registrar um gasto estrito maior que R$ 400.00, aplique o rótulo '🔴 CRÍTICO'; caso contrário, classifique como '🟢 DENTRO DO ESPERADO', exportando o relatório final direto para o Excel."*

---


In [9]:
# =================================================================================
# PIPELINES DE ENGENHARIA DE DADOS / REGRAS DE NEGÓCIO E ROTULAGEM CONDICIONAL
# CONCEITO: Uso do Pandas (.apply e Lambda) para Criação de Colunas Dinâmicas
#
# CONTEXTO DE NEGÓCIO (Pedido do Gestor):
# "Para apoiar a tomada de decisões na fase de Otimização de FinOps, preciso que o
# nosso script em Python aplique uma regra de governança automatizada no DataFrame
# antes da exportação. Adicione uma coluna chamada 'status_orcamento': se o recurso
# registrar um gasto estrito maior que R$ 400.00, aplique o rótulo '🔴 CRÍTICO';
# caso contrário, classifique como '🟢 DENTRO DO ESPERADO', exportando para o Excel."
# =================================================================================


from IPython.utils import encoding
from google.cloud.bigquery import query
from google.cloud import bigquery
import pandas as pd
import google.auth # Bliblioteca para capturar minhas credenciais
from google.colab import files
from google.colab import auth # Importando o módulo de autenticação

# Adicionando autenticação para usuário no Colab
print("Authenticating use...")
auth.authenticate_user()
print("User authenticated.")

# 1. CAPTURA DOS TOKENS: Puxa explicitamente a autenticação realizada na célula 1
crendenciais, projeto_padrao = google.auth.default()

# 2. Inicializando o cliente injetando as suas credenciasi de usuário logado
client = bigquery.Client(credentials=crendenciais, project='gen-lang-client-0862909089')

# 3. Definir a query dinâmica de subquery
query = """
SELECT
  project.id AS projeto_id,
  service.description AS servico_descricao,
  cost AS custo_bruto
FROM
  `meu_finops.custos_nuvem_bruto_gcp`
ORDER BY
  custo_bruto DESC;
"""
print("⏳ Buscando dados no BigQuery...")
# Executa a quary e converta para DataFrame do Pandas
df_custo = client.query(query).to_dataframe()

# 4. O Python cria a coluna de status de forma autônoma!
df_custo['status_orcamento'] = df_custo['custo_bruto'].apply(
    lambda x: '🔴 CRÍTICO' if x > 400.00 else '🟢 DENTRO DO ESPERADO'
)

# 5. Exibir a tela do colab para validar
print(df_custo)

# 6. Salvar e fazer dawnload
nome_arquivo = "relatorio_status_alertas.csv"
df_custo.to_csv(nome_arquivo, index=False, sep=';', encoding='utf-8-sig')
files.download(nome_arquivo)

Authenticating use...
User authenticated.
⏳ Buscando dados no BigQuery...
          projeto_id servico_descricao  custo_bruto      status_orcamento
0  ipnet-vendas-prod    Compute Engine        450.0             🔴 CRÍTICO
1    ipnet-data-lake          BigQuery        320.5  🟢 DENTRO DO ESPERADO
2  ipnet-vendas-prod     Cloud Storage         25.0  🟢 DENTRO DO ESPERADO


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>